# v5i_final · A股 ETF 轮动策略 — 一键日/周操作指引

**目标**：一个 notebook 从 0 到 1 跑通 **v5i_final**（风险平价 + 动量共振 + 高低切 + Regime 防御 + Trailing 防御），并输出：

1. **每日操作指引** — 今天该持有什么、下一个调仓日、当前 Regime/Trailing 状态；
2. **下周持仓清单** — 本周五收盘信号决定，下周一开盘执行；
3. **交易指令列表**（相对上周的买/卖/加减仓）。

### 策略核心（v5i_final）

- **选股** 风险平价（vol 过滤 + 低相关性 top-5）→ 动量共振打分 top-3；
- **权重** 1/σ 倒数加权；组合级 `vol_target=13%` 反向缩放；
- **类别上限** 商品 ≤ 0.9；
- **高低切** 进攻模式内 logBIAS + RSI 过热 → 次日换冷门候选；
- **Regime Gate** 沪深300 120 日累计 ≤ -8% → B3 多元防御篮子（金 30% + 国债 40% + 红利 15% + 煤炭 15%）；
- **Trailing Stop** 组合净值距峰值回撤 ≤ -8% → B3 篮子，直至反弹 +3% 或 20 日；
- **执行** 周五收盘信号，下周一开盘 + 单边 5bps 滑点。

### 依赖

```bash
pip install -r requirements.txt
# TUSHARE_TOKEN 已内置，仅用于 CSI300 基准
```

## 1 · Setup & Imports

In [1]:
import os, sys, time, json, warnings
from pathlib import Path
from datetime import datetime, timedelta
import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')
pd.set_option('display.max_rows', 80)
pd.set_option('display.width', 140)
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')

os.environ['TUSHARE_TOKEN'] = 'ddd1b26b20ff085ac9b60c9bd902ae76bbff60910863e8cc0168da53'

HERE  = Path.cwd()
sys.path.insert(0, str(HERE / 'src'))
CACHE = HERE / 'cache';   CACHE.mkdir(parents=True, exist_ok=True)
OUT   = HERE / 'results'; OUT.mkdir(parents=True, exist_ok=True)

def say(s): print(f'[{time.strftime("%H:%M:%S")}] {s}', flush=True)

from strategy_a_share_etf_rotation import ETF_POOL, BENCHMARK, fetch_all
from strategy_v5e_capped import ParamsV5E, run_backtest_v5e, DEFENSE_BASKETS
say(f'working dir: {HERE}')

[20:57:47] working dir: C:\Users\Hu\Downloads\RiskParity-Momentum-ETF-main


## 2 · Universe: 64 ETF 全池（宽基/成长/周期/商品/债）

In [2]:
uni_df = pd.DataFrame(
    [{'ts_code': c, 'name': n} for c, n in ETF_POOL.items()]
)
code2name = dict(zip(uni_df['ts_code'], uni_df['name']))
uni_df.to_csv(CACHE/'etf_universe.csv', index=False)
say(f'Universe: {len(uni_df)} ETFs · Benchmark: {BENCHMARK}')
uni_df.head(10)

[20:57:47] Universe: 64 ETFs · Benchmark: 510300.SH


,ts_code,name
0,510300.SH,沪深300
1,510050.SH,上证50
2,510500.SH,中证500
3,512100.SH,中证1000ETF
4,588000.SH,科创50ETF
5,588080.SH,科创板50
6,159915.SZ,创业板ETF
7,159952.SZ,创业板ETF广发
8,159949.SZ,创业板50
9,563360.SH,中证A500ETF


## 3 · 拉取 ETF 行情（akshare · parquet 缓存 · 进度条）

首次运行需 5-10 分钟；有缓存则秒级返回。

In [3]:
TODAY = pd.Timestamp.today().normalize()
START = '2015-06-01'
END   = TODAY.strftime('%Y-%m-%d')
say(f'fetch range: {START} ~ {END}')

prices = fetch_all(START, END)
say(f'price matrix: {prices.shape}  ·  latest bar: {prices.index.max().date()}')

[20:57:47] fetch range: 2015-06-01 ~ 2026-04-23


fetch:   0%|          | 0/64 [00:00<?, ?it/s]

fetch:   6%|▋         | 4/64 [00:04<01:09,  1.15s/it]

fetch:   8%|▊         | 5/64 [00:10<02:18,  2.36s/it]

fetch:   9%|▉         | 6/64 [00:19<04:07,  4.26s/it]

...

fetch:  97%|█████████▋| 62/64 [07:20<00:10,  5.48s/it]

fetch:  98%|█████████▊| 63/64 [07:25<00:05,  5.47s/it]

fetch: 100%|██████████| 64/64 [07:32<00:00,  5.77s/it]

fetch: 100%|██████████| 64/64 [07:32<00:00,  7.07s/it]

[21:05:19] price matrix: (2648, 63)  ·  latest bar: 2026-04-23


## 4 · v5i_final 信号构造 + 回测

In [4]:
# ★ 参数锁定（来自 walk-forward 4 折 expanding 选参的一致收敛结果）
# 证据链保留在 results/grid_search_3stage.csv · walk_forward.csv · walk_forward.png
# 本 notebook 不再重复搜参，每次直接用锁定参数
p = ParamsV5E(
    n_momentum=3,
    enable_weight_caps=False,
    max_category_weight=1.0,
    max_commodity_weight=0.9,
    enable_vol_targeting=True,
    vol_target=0.11,              # WF 锁定：0.13 → 0.11（回撤端改善 1.1pp）
    vol_target_window=20,
    defense_basket_name='B3_diversified',
    trailing_dd=-0.08,            # WF 锁定
    trailing_recovery=0.03,
    trailing_max_days=20,
)

SAMPLE_START = '2020-01-01'
r = run_backtest_v5e(prices, SAMPLE_START, END, p)
say(f'backtest ok · {len(r.nav)} days · {r.nav.index[0].date()} → {r.nav.index[-1].date()}')
say(f'rebalances: {len(r.rebalances)} · intraday switches: {len(r.switch_log)}')
say(f'🔒 locked params: trailing_dd={p.trailing_dd}  vol_target={p.vol_target}  basket={p.defense_basket_name}')


[21:05:24] backtest ok · 1527 days · 2020-01-02 → 2026-04-23


[21:05:24] rebalances: 292 · intraday switches: 6


[21:05:24] 🔒 locked params: trailing_dd=-0.08  vol_target=0.11  basket=B3_diversified


## 5 · 回测绩效（Full / IS / OOS）

In [5]:
def stats(nav):
    nav = nav.dropna()
    if len(nav) < 20: return {}
    rets = nav.pct_change().dropna()
    n = len(rets)
    yrs = n / 252.0
    ann = (nav.iloc[-1]/nav.iloc[0])**(1/yrs) - 1
    vol = rets.std() * np.sqrt(252)
    sh  = ann/vol if vol>0 else np.nan
    dd  = (nav/nav.cummax() - 1).min()
    cal = ann/abs(dd) if dd<0 else np.nan
    down = rets[rets<0]
    dv  = down.std()*np.sqrt(252) if len(down)>0 else np.nan
    srt = ann/dv if dv and dv>0 else np.nan
    return {'n_days':n,'ann_ret':ann,'vol':vol,'sharpe':sh,'sortino':srt,
            'max_dd':dd,'calmar':cal,'win_rate':(rets>0).mean()}

is_mask  = r.nav.index < pd.Timestamp('2024-01-01')
oos_mask = r.nav.index >= pd.Timestamp('2024-01-01')

def _rebased(nav, mask):
    sub = nav[mask].dropna()
    return sub / sub.iloc[0] if len(sub) else sub

tbl = pd.DataFrame({
    'IS  (2020-2023)': stats(_rebased(r.nav, is_mask)),
    'OOS (2024-now)':  stats(_rebased(r.nav, oos_mask)),
    'Full':            stats(r.nav),
}).T
for c in ['ann_ret','vol','max_dd','win_rate']:
    if c in tbl: tbl[c] = tbl[c].map(lambda x: f'{x*100:+.2f}%' if pd.notna(x) else '-')
for c in ['sharpe','sortino','calmar']:
    if c in tbl: tbl[c] = tbl[c].map(lambda x: f'{x:.2f}' if pd.notna(x) else '-')
tbl

,n_days,ann_ret,vol,sharpe,sortino,max_dd,calmar,win_rate
IS (2020-2023),969.0000,+19.40%,+10.31%,1.88,2.67,-6.87%,2.82,+50.77%
OOS (2024-now),556.0000,+34.96%,+11.46%,3.05,4.41,-6.77%,5.16,+57.91%
Full,"1,526.0000",+25.00%,+10.75%,2.33,3.33,-6.87%,3.64,+53.41%


In [6]:
# 分年绩效
rets = r.nav.pct_change().dropna()
py_rows = []
for yr, grp in rets.groupby(rets.index.year):
    if len(grp) < 10: continue
    ann = (1+grp).prod()**(252/len(grp)) - 1
    vol = grp.std()*np.sqrt(252)
    sh  = ann/vol if vol>0 else np.nan
    eq  = (1+grp).cumprod(); dd = (eq/eq.cummax()-1).min()
    py_rows.append({'year':yr,'ann_ret':f'{ann*100:+.2f}%','sharpe':f'{sh:.2f}',
                    'max_dd':f'{dd*100:+.2f}%','n_days':len(grp)})
py = pd.DataFrame(py_rows)
py.to_csv(OUT/'peryear.csv', index=False)
py

,year,ann_ret,sharpe,max_dd,n_days
0,2020,+28.83%,2.90,-5.02%,242
1,2021,+13.65%,1.26,-6.87%,243
2,2022,+11.25%,1.05,-5.34%,242
3,2023,+24.80%,2.55,-5.26%,242
4,2024,+25.01%,2.46,-6.07%,242
5,2025,+45.14%,3.65,-3.73%,243
6,2026,+39.87%,3.20,-6.77%,72


## 6 · 📅 每日操作指引（Daily Playbook）

本策略以**周五收盘信号 + 下周一开盘执行**为主；进攻模式下可能有日内高低切换。

In [7]:
t_last     = r.nav.index[-1]
last_daily = prices.index.max()
today      = pd.Timestamp(datetime.now().date())

def next_monday(d):
    d = pd.Timestamp(d)
    days_ahead = (0 - d.weekday()) % 7
    if days_ahead == 0: days_ahead = 7
    return d + timedelta(days=days_ahead)

# 最近周五 = 上次 weekly 调仓日
weekly_rebals = [rb for rb in r.rebalances if rb.get('kind') in ('weekly','weekly_regime','weekly_trail')]
last_fri = weekly_rebals[-1]['date'] if weekly_rebals else t_last

# 下一次调仓 = 今天之后的第一个周一；下一次信号 = 该调仓日前的周五
next_rebal = next_monday(max(last_fri, today - timedelta(days=1)))
next_signal = next_rebal - timedelta(days=3)
dow = today.day_name()

print('═'*70)
print(f'  📅 今天          : {today.date()}  ({dow})')
print(f'  📊 最近周度调仓 : {last_fri.date()}')
print(f'  📈 最新日线收盘 : {last_daily.date()}')
print(f'  🔁 下一次调仓   : {next_rebal.date()} (周一开盘)')
print(f'  💡 下一次信号   : {next_signal.date()} (周五收盘后)')
print('═'*70)

playbook = pd.DataFrame([
    {'星期':'周一','日内动作':'🛎️ 开盘按"下周持仓清单"调仓（一次性买入/调整）','频率':'每周 1 次'},
    {'星期':'周二','日内动作':'✋ 持仓不动；若进攻模式触发日内高低切则收盘后执行','频率':'可选'},
    {'星期':'周三','日内动作':'✋ 持仓不动（同上）','频率':'可选'},
    {'星期':'周四','日内动作':'✋ 持仓不动（同上）','频率':'可选'},
    {'星期':'周五','日内动作':'📝 收盘后重跑本 notebook → 生成下周一目标持仓','频率':'每周 1 次'},
    {'星期':'周末','日内动作':'🧠 复盘 Regime / Trailing 状态、vol targeting 缩尺','频率':'可选'},
])
playbook

══════════════════════════════════════════════════════════════════════
  📅 今天          : 2026-04-23  (Thursday)
  📊 最近周度调仓 : 2026-04-17
  📈 最新日线收盘 : 2026-04-23
  🔁 下一次调仓   : 2026-04-27 (周一开盘)
  💡 下一次信号   : 2026-04-24 (周五收盘后)
══════════════════════════════════════════════════════════════════════


,星期,日内动作,频率
0,周一,"🛎️ 开盘按""下周持仓清单""调仓（一次性买入/调整）",每周 1 次
1,周二,✋ 持仓不动；若进攻模式触发日内高低切则收盘后执行,可选
2,周三,✋ 持仓不动（同上）,可选
3,周四,✋ 持仓不动（同上）,可选
4,周五,📝 收盘后重跑本 notebook → 生成下周一目标持仓,每周 1 次
5,周末,🧠 复盘 Regime / Trailing 状态、vol targeting 缩尺,可选


In [8]:
# 当前 Regime + Trailing 状态
bench = prices[BENCHMARK].dropna()
reg_ret = bench.iloc[-1]/bench.iloc[-p.regime_lookback-1] - 1 if len(bench) > p.regime_lookback else np.nan
regime_off = reg_ret < p.regime_threshold

peak    = r.nav.cummax().iloc[-1]
cur_dd  = r.nav.iloc[-1]/peak - 1

last_kind = weekly_rebals[-1].get('kind') if weekly_rebals else 'weekly'
if last_kind == 'weekly_trail':
    mode = '🔴 RISK-OFF · Trailing 防御 (B3 篮子)'
elif last_kind == 'weekly_regime':
    mode = '🔴 RISK-OFF · Regime 防御 (B3 篮子)'
else:
    mode = '🟢 RISK-ON · 进攻模式 (风险平价 + 动量 top-3)'

print('═'*70)
print(f'  当前市场状态 (截至 {t_last.date()})')
print('─'*70)
print(f'  沪深300 {p.regime_lookback}日累计   : {reg_ret*100:+6.2f}%   (阈值 {p.regime_threshold*100:+.0f}%)')
print(f'  组合距峰值回撤       : {cur_dd*100:+6.2f}%   (trailing 阈值 {p.trailing_dd*100:+.0f}%)')
print(f'  Regime Gate          : {"OFF  (触发防御)" if regime_off else "ON   (正常进攻)"}')
print(f'  Latest rebalance     : {last_kind}')
print(f'  Mode                 : {mode}')
print('═'*70)

══════════════════════════════════════════════════════════════════════
  当前市场状态 (截至 2026-04-23)
──────────────────────────────────────────────────────────────────────
  沪深300 120日累计   :  +0.86%   (阈值 -8%)
  组合距峰值回撤       :  -5.12%   (trailing 阈值 -8%)
  Regime Gate          : ON   (正常进攻)
  Latest rebalance     : weekly
  Mode                 : 🟢 RISK-ON · 进攻模式 (风险平价 + 动量 top-3)
══════════════════════════════════════════════════════════════════════


## 7 · 📋 下周持仓清单（Next Week Holdings）

以最近一个周度调仓信号，生成下周一开盘应调至的目标持仓。

In [9]:
cur = weekly_rebals[-1] if weekly_rebals else None
if cur is None or not cur['target']:
    next_hold = pd.DataFrame(columns=['ts_code','ETF名称','目标权重'])
    total_w = 0.0
else:
    rows = sorted(cur['target'].items(), key=lambda x: -x[1])
    next_hold = pd.DataFrame([
        {'ts_code': c, 'ETF名称': code2name.get(c, c),
         '目标权重': f'{w*100:6.2f}%', 'weight_raw': round(w, 4)}
        for c, w in rows
    ])
    total_w = sum(w for _, w in rows)

print('═'*78)
print(f'  📋 下周一 ({next_rebal.date()}) 开盘目标持仓')
print(f'  📊 信号截止   : {last_fri.date()} 周五收盘')
print(f'  💰 总仓位     : {total_w*100:.1f}%   ·   现金: {(1-total_w)*100:.1f}%')
print(f'  ⚙️  Kind       : {last_kind}')
print('═'*78)

next_hold.to_csv(OUT/f'next_holdings_{last_fri.date()}.csv', index=False)
say(f'saved: next_holdings_{last_fri.date()}.csv')
next_hold

══════════════════════════════════════════════════════════════════════════════
  📋 下周一 (2026-04-27) 开盘目标持仓
  📊 信号截止   : 2026-04-17 周五收盘
  💰 总仓位     : 42.0%   ·   现金: 58.0%
  ⚙️  Kind       : weekly
══════════════════════════════════════════════════════════════════════════════
[21:05:24] saved: next_holdings_2026-04-17.csv


,ts_code,ETF名称,目标权重,weight_raw
0,513500.SH,标普500ETF,16.56%,0.1656
1,513100.SH,纳指ETF,13.37%,0.1337
2,159980.SZ,有色ETF大成,12.03%,0.1203


## 8 · 🔄 交易指令清单（Trade List vs 上周）

In [10]:
if len(weekly_rebals) < 2:
    trade_df = pd.DataFrame()
    prev_date = None
else:
    prev = weekly_rebals[-2]
    cur  = weekly_rebals[-1]
    prev_date = prev['date']
    prev_tgt = prev['target'] or {}
    cur_tgt  = cur['target']  or {}
    keys = set(prev_tgt) | set(cur_tgt)
    rows = []
    for k in keys:
        pw = prev_tgt.get(k, 0.0); nw = cur_tgt.get(k, 0.0)
        d = nw - pw
        if abs(d) < 1e-5: continue
        if pw == 0 and nw > 0:     side = '🟢 BUY  (新建)'
        elif nw == 0 and pw > 0:   side = '🔴 SELL (清仓)'
        elif d > 0:                side = '🟢 BUY  (加仓)'
        else:                      side = '🔴 SELL (减仓)'
        rows.append({'方向': side, 'ts_code': k, 'ETF名称': code2name.get(k, k),
                     '上周权重': f'{pw*100:+6.2f}%', '本周权重': f'{nw*100:+6.2f}%',
                     'Δ权重':  f'{d*100:+6.2f}%', '_d': d})
    trade_df = pd.DataFrame(rows)
    if not trade_df.empty:
        trade_df = trade_df.sort_values('_d', key=abs, ascending=False).drop(columns=['_d']).reset_index(drop=True)

tov_this = trade_df.shape[0]
print('═'*78)
if prev_date is not None:
    print(f'  🔄 调仓交易指令  ({prev_date.date()} → {last_fri.date()})')
print(f'  变更条目数     : {tov_this}')
print('═'*78)
if trade_df.empty:
    print('  ✅ 无需调仓（本周目标与上周相同）')
trade_df.to_csv(OUT/f'trades_{last_fri.date()}.csv', index=False)
trade_df

══════════════════════════════════════════════════════════════════════════════
  🔄 调仓交易指令  (2026-04-10 → 2026-04-17)
  变更条目数     : 3
══════════════════════════════════════════════════════════════════════════════


,方向,ts_code,ETF名称,上周权重,本周权重,Δ权重
0,🟢 BUY (新建),513500.SH,标普500ETF,+0.00%,+16.56%,+16.56%
1,🔴 SELL (减仓),513100.SH,纳指ETF,+21.85%,+13.37%,-8.48%
2,🔴 SELL (减仓),159980.SZ,有色ETF大成,+19.44%,+12.03%,-7.41%


## 9 · 📈 NAV vs 沪深300 + 超额 + Drawdown

三联图：
1. **NAV 对比**：策略 vs 沪深 300（均归一到 1.0，log 轴）
2. **超额净值**：策略 NAV / CSI300 NAV
3. **Drawdown 对比**

In [11]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
matplotlib.rcParams['font.sans-serif'] = ['SimHei','Microsoft YaHei','DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False

nav_strat = r.nav / r.nav.iloc[0]
nav_bm    = r.benchmark_nav / r.benchmark_nav.iloc[0]

common_idx = nav_strat.index.intersection(nav_bm.index)
nav_strat  = nav_strat.reindex(common_idx)
nav_bm     = nav_bm.reindex(common_idx)
excess     = nav_strat / nav_bm
dd_strat   = nav_strat/nav_strat.cummax() - 1
dd_bm      = nav_bm/nav_bm.cummax()       - 1

def _s(nav):
    rets = nav.pct_change().dropna()
    y = len(rets)/252.0
    a = (nav.iloc[-1]/nav.iloc[0])**(1/y) - 1
    v = rets.std()*np.sqrt(252)
    return a, v, a/v if v>0 else np.nan, (nav/nav.cummax() - 1).min()

a_s,v_s,sh_s,dd_s = _s(nav_strat)
a_b,v_b,sh_b,dd_b = _s(nav_bm)
a_e,v_e,sh_e,dd_e = _s(excess)

print('═'*70)
print(f'  策略 (v5i_final)    : AnnRet {a_s*100:+6.2f}%   Sharpe {sh_s:5.2f}   MaxDD {dd_s*100:+6.2f}%')
print(f'  沪深300 (benchmark) : AnnRet {a_b*100:+6.2f}%   Sharpe {sh_b:5.2f}   MaxDD {dd_b*100:+6.2f}%')
print(f'  超额 (策略/沪深300) : AnnRet {a_e*100:+6.2f}%   Sharpe {sh_e:5.2f}   MaxDD {dd_e*100:+6.2f}%')
print('═'*70)

fig, axes = plt.subplots(3, 1, figsize=(13, 10), sharex=True,
                         gridspec_kw={'height_ratios':[3,2,2]})
ax = axes[0]
ax.plot(nav_strat.index, nav_strat.values, color='#C0392B', lw=2.0,
        label=f'v5i_final (Ann {a_s*100:+.1f}%, Sh {sh_s:.2f})')
ax.plot(nav_bm.index,    nav_bm.values,    color='#2C3E50', lw=1.6, ls='--',
        label=f'CSI 300 (Ann {a_b*100:+.1f}%, Sh {sh_b:.2f})')
ax.set_yscale('log'); ax.set_ylabel('NAV (log)')
ax.set_title(f'v5i_final vs CSI 300 · {common_idx[0].date()} → {common_idx[-1].date()}',
             fontsize=13, fontweight='bold')
ax.legend(loc='upper left', fontsize=11); ax.grid(True, alpha=0.3)
ax.axhline(1.0, color='gray', lw=0.8, alpha=0.5)

ax = axes[1]
ax.plot(excess.index, excess.values, color='#27AE60', lw=1.8,
        label=f'Excess NAV (Ann {a_e*100:+.1f}%, Sh {sh_e:.2f})')
ax.fill_between(excess.index, 1.0, excess.values, where=excess.values>=1,
                color='#27AE60', alpha=0.15)
ax.fill_between(excess.index, 1.0, excess.values, where=excess.values<1,
                color='#C0392B', alpha=0.15)
ax.axhline(1.0, color='gray', lw=0.8); ax.set_ylabel('Excess NAV')
ax.legend(loc='upper left', fontsize=11); ax.grid(True, alpha=0.3)

ax = axes[2]
ax.fill_between(dd_strat.index, 0, dd_strat.values*100, color='#C0392B', alpha=0.55,
                label=f'v5i_final DD (min {dd_s*100:+.2f}%)')
ax.fill_between(dd_bm.index,    0, dd_bm.values*100,    color='#2C3E50', alpha=0.35,
                label=f'CSI 300 DD (min {dd_b*100:+.2f}%)')
ax.set_ylabel('Drawdown (%)'); ax.set_xlabel('Date')
ax.legend(loc='lower left', fontsize=11); ax.grid(True, alpha=0.3)
ax.axhline(0, color='black', lw=0.6)

plt.tight_layout()
fig_path = OUT / 'nav_vs_benchmark.png'
plt.savefig(fig_path, dpi=130, bbox_inches='tight', facecolor='white')
plt.show()
say(f'figure saved: {fig_path.name}')

cmp = pd.DataFrame({'nav_v5i': nav_strat, 'nav_csi300': nav_bm,
                    'excess': excess, 'dd_strat': dd_strat, 'dd_bm': dd_bm})
cmp.to_csv(OUT/'nav_vs_benchmark.csv')
say(f'data saved: nav_vs_benchmark.csv')

══════════════════════════════════════════════════════════════════════
  策略 (v5i_final)    : AnnRet +25.00%   Sharpe  2.33   MaxDD  -6.87%
  沪深300 (benchmark) : AnnRet  +2.47%   Sharpe  0.13   MaxDD -45.10%
  超额 (策略/沪深300) : AnnRet +21.98%   Sharpe  1.12   MaxDD -26.21%
══════════════════════════════════════════════════════════════════════


[21:05:25] figure saved: nav_vs_benchmark.png


[21:05:25] data saved: nav_vs_benchmark.csv


## 10 · 🔍 信号穿透（近 8 周调仓演化）

In [12]:
recent_rows = []
for rb in weekly_rebals[-8:]:
    tgt = rb['target'] or {'(空仓)': 0}
    for c, w in tgt.items():
        recent_rows.append({'date': rb['date'].date(), 'kind': rb.get('kind'),
                            'ts_code': c, 'ETF名称': code2name.get(c, c),
                            '权重': round(w, 4)})
recent_df = pd.DataFrame(recent_rows)
recent_df.to_csv(OUT/'recent8w_holdings.csv', index=False)
recent_df

,date,kind,ts_code,ETF名称,权重
0,2026-02-27,weekly,515220.SH,煤炭ETF国泰,0.1863
1,2026-02-27,weekly,159985.SZ,豆粕ETF华夏,0.4455
2,2026-02-27,weekly,513500.SH,标普500ETF,0.0385
3,2026-02-27,weekly,513100.SH,纳指ETF,0.0317
4,2026-03-06,weekly,515220.SH,煤炭ETF国泰,0.0976
5,2026-03-06,weekly,159985.SZ,豆粕ETF华夏,0.2196
6,2026-03-06,weekly,518880.SH,黄金ETF,0.0983
7,2026-03-06,weekly,513500.SH,标普500ETF,0.0254
8,2026-03-06,weekly,513100.SH,纳指ETF,0.0208
9,2026-03-13,weekly,518880.SH,黄金ETF,0.2091


## 11 · 💾 输出所有文件

In [13]:
# NAV & 权益曲线
pd.DataFrame({'strategy': r.nav, 'benchmark': r.benchmark_nav}).to_csv(OUT/'nav_v5i_final.csv')
# 所有调仓
rb_rows = []
for rb in r.rebalances:
    tgt = rb['target'] or {'(空仓)': 0}
    for c, w in tgt.items():
        rb_rows.append({'date': rb['date'].date(), 'kind': rb.get('kind'),
                        'ts_code': c, 'name': code2name.get(c, c),
                        'weight': w, 'nav': rb['nav']})
pd.DataFrame(rb_rows).to_csv(OUT/'rebalances_v5i_final.csv', index=False)
pd.Series(r.metrics).to_csv(OUT/'metrics_v5i_final.csv', header=['value'])

# metrics.json
meta = {
    'spec': 'v5i_final',
    'params': {'max_cmd': p.max_commodity_weight, 'vol_target': p.vol_target,
               'vt_window': p.vol_target_window, 'basket': p.defense_basket_name,
               'n_momentum': p.n_momentum, 'trailing_dd': p.trailing_dd},
    'sample': {'start': str(r.nav.index[0].date()), 'end': str(r.nav.index[-1].date()),
               'n_days': len(r.nav)},
    'metrics': r.metrics,
    'full':    stats(r.nav),
    'is':      stats(_rebased(r.nav, is_mask)),
    'oos':     stats(_rebased(r.nav, oos_mask)),
    'next_rebalance_date': str(next_rebal.date()),
    'latest_signal_date':  str(last_fri.date()),
    'mode_now': mode,
}
with open(OUT/'metrics.json', 'w', encoding='utf-8') as f:
    json.dump(meta, f, indent=2, ensure_ascii=False, default=str)

print('═'*70)
print(f'  ✅ 所有产出保存在 {OUT}')
print('─'*70)
for p_ in sorted(OUT.glob('*')):
    if p_.is_file(): print(f'    {p_.name:40s}  ({p_.stat().st_size:,} bytes)')
print('═'*70)

══════════════════════════════════════════════════════════════════════
  ✅ 所有产出保存在 C:\Users\Hu\Downloads\RiskParity-Momentum-ETF-main\results
──────────────────────────────────────────────────────────────────────
    grid_search_3stage.csv                    (901 bytes)
    grid_search_v5d_is.csv                    (2,921 bytes)
    grid_search_v5f_is.csv                    (1,241 bytes)
    grid_search_v5g_is.csv                    (1,390 bytes)
    grid_search_v5h_is.csv                    (305 bytes)
    grid_search_v5i_is.csv                    (395 bytes)
    metrics.json                              (1,520 bytes)
    metrics_v5d_final_full.csv                (155 bytes)
    metrics_v5d_final_is.csv                  (156 bytes)
    metrics_v5d_final_oos.csv                 (154 bytes)
    metrics_v5i_final.csv                     (150 bytes)
    metrics_v5i_final_full.csv                (155 bytes)
    metrics_v5i_final_is.csv                  (155 bytes)
    metrics_v5i_final_oos

## ✅ 完成

- **每日**：看 cell 6（Daily Playbook）知道今天该做什么；
- **每周五收盘**：重跑本 notebook，cell 7 即为下周一开盘目标持仓；
- **调仓**：按 cell 8 的 BUY/SELL 清单执行。

> ⚠️ 仅供量化研究参考，不构成投资建议。实盘请自行做好风控与资金管理。